In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
from collections import Counter
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
import kagglehub
import os
import json
from google.colab import userdata

kaggle_auth_json_str = userdata.get('KAGGLE_AUTH_JSON')
if kaggle_auth_json_str:
    try:
        # Create the .kaggle directory if it doesn't exist
        os.makedirs('/root/.kaggle', exist_ok=True)
        # Write the kaggle.json file
        with open('/root/.kaggle/kaggle.json', 'w') as f:
            f.write(kaggle_auth_json_str)
        # Set the file permissions
        os.chmod('/root/.kaggle/kaggle.json', 600)
        print("Kaggle API key configured successfully.")

        # Now you can download the dataset
        path = kagglehub.dataset_download("kaushal2896/english-to-german")
        print("Path to dataset files:", path)

    except json.JSONDecodeError:
        print("Error: Invalid JSON in KAGGLE_AUTH_JSON Colab secret.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("Kaggle API key not found in Colab secrets. Please add your kaggle.json content as 'KAGGLE_AUTH_JSON'.")

# Copy the downloaded files from the Kaggle cache to the content directory
# Replace 'kaushal2896/english-to-german/versions/1' with the actual path from the previous output
!cp -r /root/.cache/kagglehub/datasets/kaushal2896/english-to-german/versions/1/* /content/
print("Files copied to /content/")

Kaggle API key configured successfully.


100%|██████████| 7.99M/7.99M [00:00<00:00, 155MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kaushal2896/english-to-german/versions/1
Files copied to /content/


In [6]:
# 1. Example Data
source_sentences = ["hello how are you", "i am fine", "thank you"]
target_sentences = ["hallo wie geht es dir", "mir geht es gut", "danke"]

# 2. Tokenizer & Vocabulary Builder
def build_vocab(sentences, min_freq=1):
    counter = Counter()
    for sentence in sentences:
        tokens = word_tokenize(sentence.lower())
        counter.update(tokens)
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

src_vocab = build_vocab(source_sentences)
tgt_vocab = build_vocab(target_sentences)
inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}

def encode(sentence, vocab):
    tokens = word_tokenize(sentence.lower())
    return [vocab.get(tok, vocab["<unk>"]) for tok in tokens]

In [7]:
# 3. Dataset
class TranslationDataset(Dataset):
    def __init__(self, src, tgt, src_vocab, tgt_vocab):
        self.data = []
        for s, t in zip(src, tgt):
            src_ids = encode(s, src_vocab)
            tgt_ids = [tgt_vocab["<sos>"]] + encode(t, tgt_vocab) + [tgt_vocab["<eos>"]]
            self.data.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_lens = [len(s) for s in src_batch]
    tgt_lens = [len(t) for t in tgt_batch]
    src_max = max(src_lens)
    tgt_max = max(tgt_lens)

    src_padded = [s + [0]*(src_max - len(s)) for s in src_batch]
    tgt_padded = [t + [0]*(tgt_max - len(t)) for t in tgt_batch]

    return torch.tensor(src_padded), torch.tensor(tgt_padded)

dataset = TranslationDataset(source_sentences, target_sentences, src_vocab, tgt_vocab)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

In [8]:
# 4. Encoder-Decoder Model
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)
        return hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input, hidden):
        input = input.unsqueeze(1)  # (B) -> (B, 1)
        embedded = self.embedding(input)
        output, hidden = self.gru(embedded, hidden)
        output = self.fc(output.squeeze(1))  # (B, vocab)
        return output, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_vocab):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab = tgt_vocab

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        outputs = torch.zeros(batch_size, tgt_len, len(self.tgt_vocab))
        hidden = self.encoder(src)
        input = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden = self.decoder(input, hidden)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t] if teacher_force else top1
        return outputs



In [9]:
# 5. Instantiate and Train
embed_size = 64
hidden_size = 128
encoder = Encoder(len(src_vocab), embed_size, hidden_size)
decoder = Decoder(len(tgt_vocab), embed_size, hidden_size)
model = Seq2Seq(encoder, decoder, tgt_vocab)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(10):
    for src_batch, tgt_batch in dataloader:
        optimizer.zero_grad()
        output = model(src_batch, tgt_batch)
        output = output[:, 1:].reshape(-1, output.shape[-1])
        tgt = tgt_batch[:, 1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 6. Translate Function
def translate(sentence):
    model.eval()
    with torch.no_grad():
        src = torch.tensor([encode(sentence, src_vocab)])
        hidden = model.encoder(src)
        input = torch.tensor([tgt_vocab["<sos>"]])
        outputs = []
        for _ in range(20):
            output, hidden = model.decoder(input, hidden)
            top1 = output.argmax(1)
            if top1.item() == tgt_vocab["<eos>"]:
                break
            outputs.append(top1.item())
            input = top1
        return " ".join(inv_tgt_vocab[i] for i in outputs)

# Test
print("Translate:", translate("hello how are you"))

Epoch 1, Loss: 2.7390
Epoch 2, Loss: 0.6981
Epoch 3, Loss: 0.8060
Epoch 4, Loss: 0.5455
Epoch 5, Loss: 0.2161
Epoch 6, Loss: 0.0446
Epoch 7, Loss: 0.0315
Epoch 8, Loss: 0.0166
Epoch 9, Loss: 0.0118
Epoch 10, Loss: 0.0075
Translate: hallo wie geht es dir


In [ ]:
#@title TextCleaner
import re
import unicodedata

class TextCleaner:
    def __init__(self, lowercase: bool = True, remove_punctuation: bool = True):
        self.lowercase = lowercase
        self.remove_punctuation = remove_punctuation

    def unicode_to_ascii(self, s: str) -> str:
        """
        Normalize unicode string to ASCII.
        E.g., “Ç” -> “C”, “ñ” -> “n”
        """
        return ''.join(
            c for c in unicodedata.normalize('NFD', s)
            if unicodedata.category(c) != 'Mn'
        )

    def clean_sentence(self, sentence: str) -> str:
        # Convert unicode to ascii
        sentence = self.unicode_to_ascii(sentence)

        # Lowercase if enabled
        if self.lowercase:
            sentence = sentence.lower()

        # Remove unwanted characters (optional punctuation removal)
        if self.remove_punctuation:
            sentence = re.sub(r"[^a-zA-Z0-9]+", " ", sentence)
        else:
            sentence = re.sub(r"\s+", " ", sentence)

        # Remove extra spaces
        sentence = sentence.strip()

        return sentence

    def __call__(self, sentence: str) -> str:
        return self.clean_sentence(sentence)

In [ ]:
#@title DatasetLoader
class DatasetLoader:
    def __init__(self, path= "/content/deu.txt"):
        self.path = path
        self.cleaner = TextCleaner()

    def openFileAndCreateTextPairs(self):
        with open(self.path, "r", encoding="utf-8") as f:
            lines = f.read().strip().split("\n")
        pairs = [[line.split("\t")[0],line.split("\t")[1]] for line in lines]
        return pairs

    def cleanText(self, pairs):
        # pairs = self.openFileAndCreateTextPairs()
        pairs = [[self.cleaner(pair[0]), self.cleaner(pair[1])] for pair in pairs]
        return pairs

    def filterToMaxSize(self, pairs, maxLength = 30):
        filtered_pairs = []
        for pair in pairs:
            if len(pair[0].split()) <= maxLength and len(pair[1].split()) <= maxLength:
                filtered_pairs.append(pair)
        return filtered_pairs

    def divideToSourceAndTarget(self, pairs):
        source = [pair[0] for pair in pairs]
        target = [pair[1] for pair in pairs]
        return source, target

In [ ]:
#@title Tokenizer
class Tokenizer():
    def __init__(self):
        pass

    def build_vocab(self, sentences):
        vocab = {}
        for sentence in sentences + ['<sos>', '<eos>', '<pad>'] :
            for word in sentence.split():
                if word not in vocab:
                    vocab[word] = len(vocab)
        return vocab

    def getPadToSize(self, sentences):
        return max([len(sentence.split()) for sentence in sentences])

    def tokenize(self, vocab, sentence,padding_size):
        tokens = [vocab[word] for word in ['<sos>']+ sentence.split() + ['<pad>']*(padding_size-len(sentence.split())) + ['<eos>']]
        return tokens

    def deTokenize(self, vocab, tokens):
        sentence = [x for x in [list(vocab.keys())[list(vocab.values()).index(token)] for token in tokens] if x not in ['<sos>', '<eos>', '<pad>']]
        return " ".join(sentence)

    def tokenizeAllSentences(self):
        tokenized_sentences = []
        for sentence in self.sentences:
            tokenized_sentences.append(self.tokenize(sentence))
        return tokenized_sentences

In [ ]:
from torch.utils.data import Dataset
class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.tokenizer = Tokenizer()
        self.src_vocab = self.tokenizer.build_vocab(src_texts)
        self.src_padTo = self.tokenizer.getPadToSize(src_texts)
        self.tgt_vocab = self.tokenizer.build_vocab(tgt_texts)
        self.tgt_padTo = self.tokenizer.getPadToSize(tgt_texts)

    def __len__(self):
        return len(self.src_texts)

    def getSrcVocab(self):
        return self.src_vocab

    def getTgtVocab(self):
        return self.tgt_vocab

    def getSrcVocabSize(self):
        return len(self.src_vocab)

    def getTgtVocabSize(self):
        return len(self.tgt_vocab)

    def encode(self, vocab, sentence,padding_size):
        return self.tokenizer.tokenize(vocab, sentence,padding_size);

    def decode(self, vocab, sentence):
        return self.tokenizer.deTokenize(vocab, sentence);

    def __getitem__(self, idx):
        src_tensor = torch.tensor(self.encode(vocab = self.src_vocab, sentence = self.src_texts[idx], padding_size = self.src_padTo), dtype=torch.long)
        tgt_tensor = torch.tensor(self.encode(sentence = self.tgt_texts[idx], vocab = self.tgt_vocab, padding_size = self.tgt_padTo), dtype=torch.long)
        return src_tensor, tgt_tensor

    def getSrcEncoded(self):
        return [self.encode(vocab = self.src_vocab, sentence = sentence, padding_size = self.src_padTo) for sentence in self.src_texts]

    def getSrcEncodedToTensor(self):
        return torch.tensor(self.getSrcEncoded(), dtype=torch.long)

    def getTgtEncodedToTensor(self):
        return torch.tensor(self.getTgtEncoded(), dtype=torch.long)

    def getTgtEncoded(self):
        return [self.encode(vocab = self.tgt_vocab, sentence = sentence, padding_size = self.tgt_padTo) for sentence in self.tgt_texts]

In [ ]:
datasetLoader = DatasetLoader()
pairs = datasetLoader.openFileAndCreateTextPairs()
cleaned_pairs = datasetLoader.cleanText(pairs)
del pairs
filtered_pairs = datasetLoader.filterToMaxSize(cleaned_pairs, maxLength = 30)
del cleaned_pairs
src, tgt = datasetLoader.divideToSourceAndTarget(filtered_pairs)
del filtered_pairs
translationDataset = TranslationDataset(src, tgt)
del src
del tgt
src_encoded, tgt_encoded = translationDataset.getSrcEncodedToTensor(), translationDataset.getTgtEncodedToTensor()

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src)  # [batch_size, src_len, emb_dim]
        outputs, hidden = self.gru(embedded)  # outputs: [batch_size, src_len, hidden_dim]
        return hidden  # return final hidden state for decoder init

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, tgt, hidden):
        # tgt: [batch_size, tgt_len]
        embedded = self.embedding(tgt)  # [batch_size, tgt_len, emb_dim]
        outputs, _ = self.gru(embedded, hidden)  # outputs: [batch_size, tgt_len, hidden_dim]
        logits = self.fc_out(outputs)  # [batch_size, tgt_len, output_dim]
        return logits

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        # src: [batch_size, src_len]
        # tgt: [batch_size, tgt_len]
        hidden = self.encoder(src)
        output = self.decoder(tgt, hidden)
        return output  # [batch_size, tgt_len, output_dim]


In [ ]:
# Example dimensions
SRC_VOCAB_SIZE = translationDataset.getSrcVocabSize()
TGT_VOCAB_SIZE = translationDataset.getTgtVocabSize()
EMB_DIM = 256
HIDDEN_DIM = 512
BATCH_SIZE = 1
SRC_LEN = 32
TGT_LEN = 32

# Create model
encoder = Encoder(SRC_VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
decoder = Decoder(TGT_VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder)


In [ ]:

# Forward pass
output = model(src_encoded, tgt_encoded)